# Leakage-Safe Feature Engineering

Forecast objective: at timestamp t, use observations available through t to predict `Appliances` at t+1 (10 minutes later). Current `Appliances` is E_t and remains an input; `target_appliances_next` is E_(t+1) and is the label only.

The reusable implementation is `ml/src/feature_engineering.py`; this notebook calls that same function rather than duplicating feature logic. It does not split, scale, normalize, or train models.

In [1]:
from pathlib import Path
import sys
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidate_roots if (p / 'data/processed/energydata_cleaned.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate data/processed/energydata_cleaned.csv')
sys.path.insert(0, str(PROJECT_ROOT))
from ml.src.feature_engineering import engineer_dataset
CLEANED_PATH = PROJECT_ROOT / 'data/processed/energydata_cleaned.csv'
FEATURES_PATH = PROJECT_ROOT / 'data/processed/energydata_features.csv'
cleaned = pd.read_csv(CLEANED_PATH, parse_dates=['date'])
intervals = cleaned['date'].diff().dropna()
print(f'Cleaned input shape: {cleaned.shape}')
print(f'Missing values: {int(cleaned.isna().sum().sum())}')
print(f'Duplicate rows/timestamps: {int(cleaned.duplicated().sum())}/{int(cleaned["date"].duplicated().sum())}')
print(f'Chronological, exact 10-minute interval: {cleaned["date"].is_monotonic_increasing}, {intervals.eq(pd.Timedelta(minutes=10)).all()}')
assert cleaned.shape == (19735, 27) and cleaned.isna().sum().sum() == 0
assert cleaned.duplicated().sum() == 0 and cleaned['date'].duplicated().sum() == 0
assert cleaned['date'].is_monotonic_increasing and intervals.eq(pd.Timedelta(minutes=10)).all()

Cleaned input shape: (19735, 27)
Missing values: 0
Duplicate rows/timestamps: 0/0
Chronological, exact 10-minute interval: True, True


## Approved Feature Definitions

Temporal fields: `hour`, `day_of_week` (Monday=0 through Sunday=6), `month`, and `is_weekend` (1 on Saturday/Sunday). Cyclical fields use `sin(2*pi*hour/24)`, `cos(2*pi*hour/24)`, `sin(2*pi*day_of_week/7)`, and `cos(2*pi*day_of_week/7)`. No month cycle, holiday, season, schedule, or occupancy field is added.

Lags at origin t use only earlier Appliances observations: 1, 6, 144, and 1008 records. Rolling windows are uncentered, full windows ending at E_t: 6 and 144 observations; standard deviations use sample `ddof=1`. Environmental measurements and current `Appliances` remain values observed at t. `date` is retained for traceability but excluded from model-input columns.

The first 1008 rows are unavailable because seven days of lag history do not exist. The last row is unavailable because its t+1 target does not exist. These are supervised-sample alignment exclusions, not data cleaning; no values are imputed.

In [3]:
features, report = engineer_dataset(CLEANED_PATH, FEATURES_PATH)
input_shape = report['input_shape']
output_shape = report['output_shape']
rows_at_start = report['rows_excluded_beginning']
rows_at_end = report['rows_excluded_end']
input_checksum = report['input_checksum_before']
output_checksum = report['output_checksum']
target_column = report['target_column']
horizon = report['forecast_horizon']
print(f"Input shape: {input_shape}; final feature shape: {output_shape}")
print(f"Rows excluded at beginning/end: {rows_at_start}/{rows_at_end}")
print(f"Input SHA-256: {input_checksum}")
print(f"Feature CSV SHA-256: {output_checksum}")
print(f"Temporal features: {report['temporal_features']}")
print(f"Cyclical features: {report['cyclical_features']}")
print(f"Lag features: {report['lag_features']}")
print(f"Rolling features: {report['rolling_features']}")
print(f"Target: {target_column}; horizon: {horizon}")
print('Leakage/alignment checks:')
for check, passed in report['alignment_checks'].items():
    print(f"  {check}: {passed}")
assert all(report['alignment_checks'].values())

Input shape: (19735, 27); final feature shape: (18726, 44)
Rows excluded at beginning/end: 1008/1
Input SHA-256: d4d5964a61a5a6ddfc1148012abb77f14f826dfa0ebd6f9b5e52d93a6c705a19
Feature CSV SHA-256: f69480c282e32601920f0820cdf9d2530ecfc5b053c4cefa4de2d15c73c2077f
Temporal features: ['hour', 'day_of_week', 'month', 'is_weekend']
Cyclical features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
Lag features: ['appliances_lag_1', 'appliances_lag_6', 'appliances_lag_144', 'appliances_lag_1008']
Rolling features: ['appliances_roll_mean_1h', 'appliances_roll_std_1h', 'appliances_roll_mean_24h', 'appliances_roll_std_24h']
Target: target_appliances_next; horizon: 0 days 00:10:00
Leakage/alignment checks:
  next_target_equals_original_next_observation: True
  appliances_lag_1_uses_history_only: True
  appliances_lag_6_uses_history_only: True
  appliances_lag_144_uses_history_only: True
  appliances_lag_1008_uses_history_only: True
  rolling_1h_is_backward_looking: True
  rolling_24h_is_backward

## Inspect Alignment and Final Schema

Each example row pairs the current value and historical lag at forecast origin t with the target value at the next timestamp. The output keeps the original physical measurements plus only the approved derived columns.

In [5]:
alignment_view = features[['date', 'Appliances', 'appliances_lag_1', 'target_appliances_next']].head(6)
display(alignment_view)
print(f'Final columns ({len(features.columns)}): {features.columns.tolist()}')
print('Final dtypes:')
display(features.dtypes)
expected_rows = len(cleaned) - report['rows_excluded_beginning'] - report['rows_excluded_end']
expected_columns = len(cleaned.columns) + len(report['temporal_features']) + len(report['cyclical_features']) + len(report['lag_features']) + len(report['rolling_features']) + 1
assert features.shape == (expected_rows, expected_columns)
assert report['rows_excluded_beginning'] == 1008 and report['rows_excluded_end'] == 1
assert features['target_appliances_next'].notna().all() and features.isna().sum().sum() == 0
assert features['date'].is_monotonic_increasing and features['date'].duplicated().sum() == 0
assert report['input_checksum_before'] == report['input_checksum_after']
assert all(report['alignment_checks'].values())
print('Derived schema, row alignment, missingness, chronology, and input-integrity checks passed.')

,date,Appliances,appliances_lag_1,target_appliances_next
0,2016-01-18 17:00:00,40,50,40
1,2016-01-18 17:10:00,40,40,40
2,2016-01-18 17:20:00,40,40,40
3,2016-01-18 17:30:00,40,40,20
4,2016-01-18 17:40:00,20,40,30
5,2016-01-18 17:50:00,30,20,30


Final columns (44): ['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'hour', 'day_of_week', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'appliances_lag_1', 'appliances_lag_6', 'appliances_lag_144', 'appliances_lag_1008', 'appliances_roll_mean_1h', 'appliances_roll_std_1h', 'appliances_roll_mean_24h', 'appliances_roll_std_24h', 'target_appliances_next']
Final dtypes:


date                        datetime64[us]
Appliances                           int64
lights                               int64
T1                                 float64
RH_1                               float64
T2                                 float64
RH_2                               float64
T3                                 float64
RH_3                               float64
T4                                 float64
RH_4                               float64
T5                                 float64
RH_5                               float64
T6                                 float64
RH_6                               float64
T7                                 float64
RH_7                               float64
T8                                 float64
RH_8                               float64
T9                                 float64
RH_9                               float64
T_out                              float64
Press_mm_hg                        float64
RH_out     

Derived schema, row alignment, missingness, chronology, and input-integrity checks passed.
